In [ ]:
import sqlite3
import csv
import os

DB_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "omni_pundit.db")
EXPORT_DIR = os.path.join(os.path.dirname(__file__), "..", "data", "exports")


def export_table(cursor, table_name, exclude_columns=None, filename_override=None):
    """Streams a table out to CSV row-by-row instead of loading it all into
    memory at once - safe for a 212K+ row table."""
    exclude_columns = exclude_columns or []

    cursor.execute(f"PRAGMA table_info({table_name})")
    all_columns = [row[1] for row in cursor.fetchall()]
    columns = [c for c in all_columns if c not in exclude_columns]

    out_name = filename_override or f"{table_name}.csv"
    out_path = os.path.join(EXPORT_DIR, out_name)
    col_list = ", ".join(columns)

    cursor.execute(f"SELECT {col_list} FROM {table_name}")

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        row_count = 0
        for row in cursor:
            writer.writerow(row)
            row_count += 1

    print(f"  -> {out_name}: {row_count:,} rows, columns: {columns}")


def export_all():
    if not os.path.exists(DB_PATH):
        print(f"ERROR: Database not found at {os.path.abspath(DB_PATH)}")
        return

    os.makedirs(EXPORT_DIR, exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [row[0] for row in cursor.fetchall()]
    print(f"Found tables: {tables}")

    for table in tables:
        # chess_games.pgn is the big text blob that makes the raw table huge
        # and unusable in Excel - export it as its own file (or skip it
        # entirely if you don't need it) so the main CSV stays small and
        # actually opens.
        if table == "chess_games":
            print(f"Exporting '{table}' (metadata only, pgn excluded)...")
            export_table(cursor, table, exclude_columns=["pgn"])

            export_pgn = input(
                "Also export raw PGN text separately? This file will be large. (y/n): "
            ).strip().lower()
            if export_pgn == "y":
                export_table(cursor, table, filename_override="chess_games_with_pgn.csv")
        else:
            print(f"Exporting '{table}'...")
            export_table(cursor, table)

    conn.close()
    print(f"\nDone. Files are in: {os.path.abspath(EXPORT_DIR)}")


if __name__ == "__main__":
    export_all()

In [ ]:
import sqlite3
import chess
import chess.engine
import chess.pgn
import io
import os
import sys

# Force Python to print everything to the terminal instantly without buffering
sys.stdout.reconfigure(line_buffering=True)

DB_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "omni_pundit.db")

# Automatically add .exe if running on Windows
STOCKFISH_EXECUTABLE = "stockfish.exe" if os.name == 'nt' else "stockfish"
STOCKFISH_PATH = os.path.join(os.path.dirname(__file__), STOCKFISH_EXECUTABLE)

def check_database_status():
    print(f"Checking database at: {os.path.abspath(DB_PATH)}")
    if not os.path.exists(DB_PATH):
        print("ERROR: Database file does not exist yet! Run chess_ingest.py first.")
        return False
        
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    cursor.execute("SELECT COUNT(*) FROM chess_games")
    total = cursor.fetchone()[0]
    print(f"Total games stored in database: {total}")
    
    try:
        cursor.execute("SELECT COUNT(*) FROM chess_games WHERE white_acpl IS NULL")
        unanalyzed = cursor.fetchone()[0]
        print(f"Games waiting to be analyzed: {unanalyzed}")
    except sqlite3.OperationalError:
        unanalyzed = total
        
    conn.close()
    return unanalyzed > 0

def setup_analysis_columns():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    try:
        cursor.execute("ALTER TABLE chess_games ADD COLUMN white_acpl INTEGER")
        cursor.execute("ALTER TABLE chess_games ADD COLUMN black_acpl INTEGER")
        print("Added new analytical columns to database layout.")
    except sqlite3.OperationalError:
        pass
    conn.commit()
    conn.close()

# Cap any single move's centipawn loss before averaging. Without this, one
# blunder into a forced mate (mate_score=10000) makes that game's ACPL a huge
# outlier that no longer reflects "how well did this player play" - it just
# reflects "did they miss one mate." Lichess and most ACPL implementations
# clamp per-move loss for this reason.
MAX_CP_LOSS_PER_MOVE = 1000

# Depth-based analysis instead of a wall-clock time budget. time=0.05 means
# "however many nodes Stockfish can search in 50ms," which depends on what
# else your machine is doing at that moment - so the same game analyzed twice
# can produce different ACPL numbers. Since ACPL is your core signal for
# player form, it needs to be reproducible. A fixed depth searches the same
# amount regardless of system load. Depth 14 is a reasonable local-CPU
# tradeoff between accuracy and speed for batch processing.
ANALYSIS_DEPTH = 14

def analyze_game(pgn_text, engine):
    game = chess.pgn.read_game(io.StringIO(pgn_text))
    
    if game is None:
        return -1, -1
        
    # THE FIX: Check if the game is a weird variant. If so, skip it!
    game_variant = game.headers.get("Variant", "Standard")
    if game_variant != "Standard":
        print(f"  -> Skipping game: Stockfish does not support the '{game_variant}' variant.")
        # Return -1 so the database marks it as "processed" and doesn't get stuck on it
        return -1, -1
        
    board = game.board()
    white_loss = 0
    black_loss = 0
    white_moves = 0
    black_moves = 0
    
    limit = chess.engine.Limit(depth=ANALYSIS_DEPTH)
    
    info = engine.analyse(board, limit)
    prev_eval = info["score"].white().score(mate_score=10000)
    
    for move in game.mainline_moves():
        is_white_turn = board.turn == chess.WHITE
        board.push(move)
        
        info = engine.analyse(board, limit)
        current_eval = info["score"].white().score(mate_score=10000)
        
        if current_eval is not None and prev_eval is not None:
            if is_white_turn:
                cp_loss = max(0, prev_eval - current_eval)
            else:
                cp_loss = max(0, current_eval - prev_eval)
            
            # Clamp before accumulating - see MAX_CP_LOSS_PER_MOVE comment above.
            cp_loss = min(cp_loss, MAX_CP_LOSS_PER_MOVE)
            
            if is_white_turn:
                white_loss += cp_loss
                white_moves += 1
            else:
                black_loss += cp_loss
                black_moves += 1
                
        prev_eval = current_eval
        
    white_acpl = int(white_loss / white_moves) if white_moves > 0 else 0
    black_acpl = int(black_loss / black_moves) if black_moves > 0 else 0
    
    return white_acpl, black_acpl

def run_batch_analysis():
    print("--- Starting Omni-Pundit Analysis Engine ---")
    
    has_games = check_database_status()
    setup_analysis_columns()
    
    if not os.path.exists(STOCKFISH_PATH):
        print(f"\nERROR: Stockfish binary missing!")
        return

    if not has_games:
        print("Exiting: No games require processing right now.")
        return
        
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    
    # Analyze in batches of 5
    cursor.execute("SELECT id, white_player, black_player, pgn FROM chess_games WHERE white_acpl IS NULL LIMIT 5")
    unprocessed_games = cursor.fetchall()
    
    print(f"Connecting to Stockfish engine...")
    try:
        with chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH) as engine:
            # Default UCI options are 1 thread / 16MB hash - fine for a quick
            # test but wasteful for batch runs on a local machine. Bumping
            # these gives Stockfish more search speed and memory for
            # transposition tables, for free, since you're not paying for
            # cloud compute. Tune Threads down if your machine has fewer
            # cores available.
            engine.configure({"Threads": 4, "Hash": 256})
            for game_id, white, black, pgn in unprocessed_games:
                print(f"Analyzing Match: {white} vs {black}...")
                w_acpl, b_acpl = analyze_game(pgn, engine)
                
                if w_acpl is not None:
                    cursor.execute("""
                        UPDATE chess_games 
                        SET white_acpl = ?, black_acpl = ? 
                        WHERE id = ?
                    """, (w_acpl, b_acpl, game_id))
                    conn.commit()
                    
                    if w_acpl != -1:
                        print(f"  -> SUCCESS! White Accuracy (ACPL): {w_acpl} | Black Accuracy (ACPL): {b_acpl}")
    except Exception as e:
        print(f"Engine Exception occurred: {e}")
    finally:
        conn.close()
        
    print("--- Batch Analysis Complete ---")

if __name__ == "__main__":
    run_batch_analysis()

In [ ]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
import os

# Handle environments where __file__ is not defined (like Jupyter/IPython)
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()

DB_PATH = os.path.abspath(os.path.join(base_dir, "..", "data", "omni_pundit.db"))

class DBViewerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Omni-Pundit Local DB Explorer")
        self.root.geometry("1000x600")
        
        # Check DB
        if not os.path.exists(DB_PATH):
            messagebox.showerror("Error", f"Database not found at {DB_PATH}")
            return
            
        self.conn = sqlite3.connect(DB_PATH)
        
        # Top Frame (Controls)
        top_frame = tk.Frame(root, pady=10, padx=10)
        top_frame.pack(fill=tk.X)
        
        tk.Label(top_frame, text="Select Table:", font=("Arial", 12)).pack(side=tk.LEFT)
        
        self.table_var = tk.StringVar()
        self.table_dropdown = ttk.Combobox(top_frame, textvariable=self.table_var, state="readonly", width=30)
        self.table_dropdown.pack(side=tk.LEFT, padx=10)
        self.table_dropdown.bind("<<ComboboxSelected>>", self.load_table_data)
        
        tk.Label(top_frame, text="(Showing first 1,000 rows)", fg="gray").pack(side=tk.LEFT, padx=10)
        
        # Refresh Button
        ttk.Button(top_frame, text="Refresh Tables", command=self.load_tables).pack(side=tk.RIGHT)
        
        # Treeview (Data Grid)
        self.tree_frame = tk.Frame(root)
        self.tree_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        self.tree = ttk.Treeview(self.tree_frame, show='headings')
        
        # Scrollbars
        vsb = ttk.Scrollbar(self.tree_frame, orient="vertical", command=self.tree.yview)
        hsb = ttk.Scrollbar(self.tree_frame, orient="horizontal", command=self.tree.xview)
        self.tree.configure(yscrollcommand=vsb.set, xscrollcommand=hsb.set)
        
        self.tree.grid(column=0, row=0, sticky='nsew')
        vsb.grid(column=1, row=0, sticky='ns')
        hsb.grid(column=0, row=1, sticky='ew')
        self.tree_frame.grid_columnconfigure(0, weight=1)
        self.tree_frame.grid_rowconfigure(0, weight=1)
        
        self.load_tables()
        
    def load_tables(self):
        cursor = self.conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [row[0] for row in cursor.fetchall()]
        self.table_dropdown['values'] = tables
        if tables:
            self.table_dropdown.current(0)
            self.load_table_data()
            
    def load_table_data(self, event=None):
        table_name = self.table_var.get()
        if not table_name: return
        
        # Clear existing data
        self.tree.delete(*self.tree.get_children())
        
        cursor = self.conn.cursor()
        
        # Get Columns
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = [col[1] for col in cursor.fetchall()]
        
        self.tree["columns"] = columns
        for col in columns:
            self.tree.heading(col, text=col)
            self.tree.column(col, width=120, anchor=tk.CENTER)
            
        # Get Data (Limit 1000 for safety on 8GB RAM)
        cursor.execute(f"SELECT * FROM {table_name} LIMIT 1000")
        rows = cursor.fetchall()
        
        for row in rows:
            self.tree.insert("", tk.END, values=row)
            

if __name__ == "__main__":
    root = tk.Tk()
    app = DBViewerApp(root)
    root.mainloop()